# Data Passport — Datenprüfung

Datenpass der bereinigten Datensätze. Wird nach den Notebooks 01–04 ausgeführt.  
Ziel: Sicherstellen, dass alle Quellen korrekt geladen werden und die Verknüpfungen zwischen ihnen funktionieren.

In [1]:
import pandas as pd
import os

import help_130625_dam as h

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

CLEANED_DIR = os.path.join('..', 'data', 'cleaned')

## Laden der bereinigten Daten

In [ ]:
# Alle .pkl-Dateien laden
contacts = pd.read_pickle(os.path.join(CLEANED_DIR, 'contacts_clean.pkl'))
deals = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
calls = pd.read_pickle(os.path.join(CLEANED_DIR, 'calls_clean.pkl'))
spend = pd.read_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))

print(f"Contacts: {contacts.shape}")
print(f"Deals: {deals.shape}")
print(f"Calls: {calls.shape}")
print(f"Spend: {spend.shape}")

Contacts: (18510, 7)
Deals: (19815, 25)
Calls: (92599, 9)
Spend: (19862, 8)


## Übersicht der Datensätze

In [ ]:
datasets = {
    'contacts': contacts,
    'deals':    deals,
    'calls':    calls,
    'spend':    spend,
}

for name, df in datasets.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    null_str = ', '.join(f'{c}: {n}' for c, n in nulls.items()) if len(nulls) else 'keine'
    print(f"\n{'─'*55}")
    print(f"  {name.upper()}: {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten")
    print(f"  Fehlende Werte: {null_str}")
    print(f"  Zeitraum: {df.select_dtypes('datetime').apply(lambda s: f'{s.min().date()} → {s.max().date()}').to_dict()}")


───────────────────────────────────────────────────────
  CONTACTS: 18,510 строк × 7 столбцов
  Пропуски: first_payment_date: 17686, new_registration_date: 18487
  Период: {'created_time': '2023-06-27 → 2024-06-21', 'modified_time': '2023-07-06 → 2024-06-21', 'first_payment_date': '2023-07-04 → 2024-06-15', 'new_registration_date': '2023-07-03 → 2024-04-29'}

───────────────────────────────────────────────────────
  DEALS: 19,815 строк × 25 столбцов
  Пропуски: closing_date: 5040
  Период: {'closing_date': '2022-10-11 → 2024-12-11', 'created_time': '2023-07-03 → 2024-06-21'}

───────────────────────────────────────────────────────
  CALLS: 92,599 строк × 9 столбцов
  Пропуски: нет
  Период: {'call_start_time': '2023-06-30 → 2024-06-21', 'call_end_time': '2023-06-30 → 2024-06-21'}

───────────────────────────────────────────────────────
  SPEND: 19,862 строк × 8 столбцов
  Пропуски: нет
  Период: {'date': '2023-07-03 → 2024-06-21'}


## Verknüpfungsprüfung (Cross-check)

In [ ]:
contact_ids = set(contacts['id'])

# Deals → Contacts (ohne -1: Deals ohne Kontakt)
valid_deals = deals[deals['contact_id'] != -1]
deals_matched = valid_deals['contact_id'].isin(contact_ids).sum()
print(f"Deals → Contacts:  {deals_matched:,} / {len(valid_deals):,} ({deals_matched/len(valid_deals)*100:.1f}%)  "
      f"[+ {(deals['contact_id'] == -1).sum():,} Deals ohne contact_id]")

# Calls → Contacts (ohne -1: unbekannte Kontakte)
valid_calls = calls[calls['contactid'] != -1]
calls_matched = valid_calls['contactid'].isin(contact_ids).sum()
print(f"Calls → Contacts:  {calls_matched:,} / {len(valid_calls):,} ({calls_matched/len(valid_calls)*100:.1f}%)  "
      f"[+ {(calls['contactid'] == -1).sum():,} Anrufe ohne contactid]")

Deals → Contacts:  19,768 / 19,768 (100.0%)  [+ 47 сделок без contact_id]
Calls → Contacts:  88,800 / 88,800 (100.0%)  [+ 3,799 звонков без contactid]
